In [8]:
# =========================
# Task 1: Load, Inspect, and Clean
# =========================

import pandas as pd
import numpy as np
import os
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Download dataset from KaggleHub
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")
print("Path to dataset files:", path)

# Check available files
print("Files in dataset folder:", os.listdir(path))

# Load CSV
file_path = os.path.join(path, "cardekho_dataset.csv")
df = pd.read_csv(file_path)

# Print initial shape
print("Initial shape:", df.shape)

# Print info
print("\nDataFrame info:")
df.info()

# Print missing percentage
print("\nMissing percentage by column:")
print(df.isnull().sum() * 100 / len(df))


# Drop rows where selling_price is missing
df = df.dropna(subset=["selling_price"])

# Clean mileage, engine, and max_power
cols_to_clean = ["mileage", "engine", "max_power"]

for col in cols_to_clean:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[^0-9.]", "", regex=True)   # remove units like kmpl, CC, bhp, etc.
        .replace("", np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Remove invalid / unrealistic selling_price values
df = df[(df["selling_price"] != 999999999) & (df["selling_price"] >= 10000)]

# Drop duplicate rows
df = df.drop_duplicates()

# Print shape after cleaning
print("\nShape after cleaning:", df.shape)


# =========================
# Task 2: Encode Categorical Features
# =========================

# Label encode transmission: Manual -> 0, Automatic -> 1
df["transmission_type"] = df["transmission_type"].map({
    "Manual": 0,
    "Automatic": 1
})

# One-hot encode fuel and seller_type
df = pd.get_dummies(df, columns=["fuel_type", "seller_type"], drop_first=True) # Corrected fuel to fuel_type

# Print final column list
print("\nFinal columns:")
print(df.columns.tolist())


# =========================
# Task 3: Split and Compute Baseline MAE
# =========================

# Define X and y
X = df.drop(columns=["selling_price"])
y = df["selling_price"]

# Drop any remaining non-numeric columns from X
X = X.select_dtypes(include=[np.number])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Baseline prediction: mean of y_train
baseline_pred = np.full(len(y_test), y_train.mean())

# Baseline MAE
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"\nBaseline MAE: ₹{round(baseline_mae):,}")

Using Colab cache for faster access to the 'cardekho-used-car-data' dataset.
Path to dataset files: /kaggle/input/cardekho-used-car-data
Files in dataset folder: ['cardekho_dataset.csv']
Initial shape: (15411, 14)

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-nu

## Why is `drop_first=True` used in one-hot encoding?

`drop_first=True` is used to remove one dummy column from each categorical feature to avoid redundant information and perfect multicollinearity. If all dummy columns are kept, one category can always be inferred from the others, which can create problems for models like linear regression.

## What does a row of all zeros in the new dummy columns represent?

A row with all zeros in the dummy columns represents the category that was dropped during one-hot encoding. That dropped category becomes the baseline or reference category.